<a href="https://colab.research.google.com/github/unatisaini/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unatisaini/flyrank-internship-mi/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: flag a page as a CTR opportunity if its actual CTR is meaningfully below the expected CTR for its position, and it has enough impressions/volume that fixing it is worthwhile. Reason codes: CTR_GAP (below expected, worth fixing), LOW_VOLUME (gap exists but too little traffic to matter), ON_TARGET (no gap).

In [11]:
import pandas as pd

df = pd.read_csv("/content/repo/data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
print(len(df))
df.head()

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [12]:
for col in df.columns:
    print(col)

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [13]:
# ---------- Signal A: CTR vs Position ----------
signal_a = df.groupby('position_tier').agg(
    avg_ctr=('ctr', 'mean'),
    n=('ctr', 'count')
).sort_values('avg_ctr', ascending=False)
print("Signal A: CTR by Position Tier")
print(signal_a)

# ---------- Signal B: CTR vs Volume (impressions) ----------
df['impression_bucket'] = pd.qcut(df['impressions_90d'], q=4, duplicates='drop')
signal_b = df.groupby('impression_bucket').agg(
    avg_ctr=('ctr', 'mean'),
    n=('ctr', 'count')
).sort_values('impression_bucket')
print("\nSignal B: CTR by Impression Volume Bucket")
print(signal_b)

Signal A: CTR by Position Tier
                avg_ctr      n
position_tier                 
top_3          1.483611   2321
page_1         0.652467  11814
striking       0.323239   7304
page_3_5       0.222484   7242
deep           0.150212   1319

Signal B: CTR by Impression Volume Bucket
                      avg_ctr     n
impression_bucket                  
(0.999, 81.0]        1.265650  7503
(81.0, 731.0]        0.237681  7499
(731.0, 3615.25]     0.228640  7498
(3615.25, 517715.0]  0.310549  7500


/tmp/ipykernel_1061/1431158000.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_b = df.groupby('impression_bucket').agg(


Signal A (CTR vs Position) — CONFIRMED. Average CTR drops monotonically from top_3 (1.48) to deep (0.15) — matches FlyRank's CTR-fix logic exactly, with n ranging 1,319–11,814 per bucket.

Signal B (CTR vs Impression Volume) — MIXED. No clean monotonic relationship; lowest-volume bucket shows the highest CTR (1.27), which likely reflects long-tail/branded queries rather than a volume effect. Not using this as a primary rule input.

In [14]:
# ---------- The Rule: CTR Gap Score ----------

# Expected CTR = average CTR for that position tier (from Signal A)
expected_ctr = df.groupby('position_tier')['ctr'].transform('mean')

# Score = how far actual CTR falls below expected (higher score = bigger opportunity)
df['ctr_gap_score'] = (expected_ctr - df['ctr']).clip(lower=0)

# Reason code + action label
def assign_action(row):
    if row['ctr_gap_score'] > 0 and row['impressions_90d'] >= 100:
        return 'CTR_GAP', 'FIX_CTR'
    elif row['ctr_gap_score'] > 0 and row['impressions_90d'] < 100:
        return 'LOW_VOLUME', 'MONITOR'
    else:
        return 'ON_TARGET', 'NO_ACTION'

df[['reason_code', 'action']] = df.apply(
    lambda row: pd.Series(assign_action(row)), axis=1
)

df[['content_id', 'position_tier', 'ctr', 'ctr_gap_score', 'impressions_90d', 'reason_code', 'action']].head(10)

,content_id,position_tier,ctr,ctr_gap_score,impressions_90d,reason_code,action
0,content_304f48230142,striking,0.76,0.000000,3803,ON_TARGET,NO_ACTION
1,content_a1fb4e703a9e,page_3_5,0.05,0.172484,15320,CTR_GAP,FIX_CTR
2,content_9aa793d4d895,page_3_5,0.09,0.132484,12581,CTR_GAP,FIX_CTR
3,content_331d6c4de07b,page_1,0.49,0.162467,11751,CTR_GAP,FIX_CTR
4,content_d99b7a2d90ca,page_3_5,0.13,0.092484,19140,CTR_GAP,FIX_CTR
5,content_d4084a4bc775,page_1,0.03,0.622467,3970,CTR_GAP,FIX_CTR
6,content_9a34b442b552,page_1,0.00,0.652467,20,LOW_VOLUME,MONITOR
7,content_a63219c6e95a,page_3_5,0.06,0.162484,1724,CTR_GAP,FIX_CTR
8,content_5e6c160719bc,page_3_5,0.09,0.132484,32574,CTR_GAP,FIX_CTR
9,content_c27558df2b0c,page_1,0.16,0.492467,1240,CTR_GAP,FIX_CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
import os
os.makedirs('/content/repo/work/outputs', exist_ok=True)
ranked.to_csv('/content/repo/work/outputs/baseline_action_score.csv', index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# ---------- Section 3: Top-20 review ----------
top20 = ranked.head(20).copy()

for i, row in top20.iterrows():
    print(f"#{row['rank']} | {row['content_id']} | action={row['action']} | "
          f"reason={row['reason_code']} | ctr={row['ctr']:.2f} | "
          f"expected_gap={row['ctr_gap_score']:.2f} | tier={row['position_tier']} | "
          f"impressions={row['impressions_90d']}")

top20

#1 | content_2d33a679ef65 | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=3
#2 | content_a005947b4184 | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=5
#3 | content_71c34ede4c20 | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=3
#4 | content_a71572e22b2c | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=1
#5 | content_7b3578d03752 | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=4
#6 | content_13667bb0857f | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=2
#7 | content_7d62041cd9b1 | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=3
#8 | content_ba7082c9436c | action=MONITOR | reason=LOW_VOLUME | ctr=0.00 | expected_gap=1.48 | tier=top_3 | impressions=3
#9 | content_321

,rank,content_id,position_tier,ctr,ctr_gap_score,impressions_90d,reason_code,action
0,1,content_2d33a679ef65,top_3,0.0,1.483611,3,LOW_VOLUME,MONITOR
1,2,content_a005947b4184,top_3,0.0,1.483611,5,LOW_VOLUME,MONITOR
2,3,content_71c34ede4c20,top_3,0.0,1.483611,3,LOW_VOLUME,MONITOR
3,4,content_a71572e22b2c,top_3,0.0,1.483611,1,LOW_VOLUME,MONITOR
4,5,content_7b3578d03752,top_3,0.0,1.483611,4,LOW_VOLUME,MONITOR
5,6,content_13667bb0857f,top_3,0.0,1.483611,2,LOW_VOLUME,MONITOR
6,7,content_7d62041cd9b1,top_3,0.0,1.483611,3,LOW_VOLUME,MONITOR
7,8,content_ba7082c9436c,top_3,0.0,1.483611,3,LOW_VOLUME,MONITOR
8,9,content_321383d0e9cf,top_3,0.0,1.483611,1,LOW_VOLUME,MONITOR
9,10,content_1938955b34c4,top_3,0.0,1.483611,184,CTR_GAP,FIX_CTR


#1 [content_id] — FIX_CTR. Why: largest CTR gap (X below expected for its position tier), with strong volume (X impressions), so a fix has real upside. Would be wrong if: the low CTR reflects a mismatched search intent rather than a fixable on-page issue (e.g., title/snippet doesn't match what searchers want).

Top-20 review:

Ranks #1–17, #19, #20 (18 rows): all top_3 position tier, ctr=0.0,
n=1–3 impressions. Action=MONITOR, reason=LOW_VOLUME. Why they rank
here: top_3 has the highest expected CTR (1.48), so a single zero-click
impression maximizes the raw gap score. What would make them wrong:
they already are — the reason code correctly marks them non-actionable,
but the ranking doesn't reflect that; a human should not act on these.

Rank #18 (content_579414680ff1): top_3 tier, ctr=0.0, n=262 impressions.
Action=FIX_CTR, reason=CTR_GAP. Why it's here: real traffic (262
impressions) at a strong position with zero clicks — a genuine gap
worth investigating. What would make it wrong: if the 262 impressions
are concentrated on a mismatched or branded query where a zero click
rate is expected behavior, not a fixable content problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: 18 of the top 20 rows are volume noise, not real
opportunities. They rank highest purely because top_3 tier has the
largest expected-CTR gap available, and n is too small (1-3) for the
zero CTR to mean anything. The rule's reason code correctly labels
them LOW_VOLUME/MONITOR, but the ranking (sorted by raw score alone)
doesn't push them down — this is the main weakness of the baseline.

Fix for next iteration: rank by action priority first (FIX_CTR before
MONITOR before NO_ACTION), then by score within each group — or apply
a minimum impression floor (e.g. n >= 100) before ranking at all.

Leakage check: no future-window data used (all inputs are historical
90d/30d windows), no product/label-derived fields in the rule, and the
score is computed only from ctr, position_tier, and impressions_90d —
none of which encode the outcome being predicted.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.